In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
print(os.cpu_count())

144


In [10]:
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
import random as rand
from statistics import mean, median,variance,stdev
import csv

from qulacs import QuantumState,QuantumCircuit, Observable, PauliOperator,ParametricQuantumCircuit,DensityMatrix, GeneralQuantumOperator
from qulacs.gate import H,X,Y,Z,RX,RY,RZ,CNOT,CZ,TOFFOLI,SWAP,merge,DenseMatrix,add, Probabilistic, P0, P1,to_matrix_gate,PauliRotation,Pauli, RandomUnitary
from qulacs.state import inner_product, partial_trace, tensor_product

# Quantum circuit

In [3]:
# observable
obsX = GeneralQuantumOperator(2)
obsX.add_operator( PauliOperator("X 0 X 1", 1.0) )

In [4]:
def random_initialize(params):
    for i in range(len(params)):
        params[i] = 2 * np.pi * rand.random()

In [5]:
def my_update(self, params):
    
    for i in range(len(params)):
        self.set_parameter(i, params[i])

ParametricQuantumCircuit.my_update = my_update

In [6]:
# See section "Quantum circuits used in numerical experiments" in Methods of the paper
# n represents the number of qubits
# d represents the depth of the symmetric ansatz, where the number of rotation gates is L=d*n*/2.
def make_circ_sc(n, d):

    circ = ParametricQuantumCircuit(n)
    for i in range(d):
        k = i%6
        if k%2==0:
            for j in range(n//2):
                mu = j + k//2
                target = [(2*j)%n, (2*j+1)%n]
                if mu%3 == 0:
                    pauli = [1,1]
                elif mu%3 == 1:
                    pauli = [2,2]
                else:
                    pauli = [3,3]
                circ.add_parametric_multi_Pauli_rotation_gate(target, pauli, 0)
                #print("target = ",target, "pauli = ",pauli)
                    
        elif k%2==1:
            for j in range(n//2):
                nu = j + (k-1)//2 + n//2
                target = [(2*j+1)%n, (2*j+2)%n]
                if nu%3 == 0:
                    pauli = [1,1]
                elif nu%3 == 1:
                    pauli = [2,2]
                else:
                    pauli = [3,3]
                circ.add_parametric_multi_Pauli_rotation_gate(target, pauli, 0)
                #print("target = ",target, "pauli = ",pauli)
                      
            #print("target = ", target)
            #print("pauli = ", pauli)
                
    return circ

# Calculate variance

In [7]:
def cal_var_sc(n, d, ns):
    num_params = d * n // 2
    params = np.zeros(num_params)
    random_initialize(params)
    
    circ = make_circ_sc(n, d)
    
    results = []
    for _ in range(ns):
        random_initialize(params)
        circ.my_update(params)
        qstate = QuantumState(n)
        circ.update_quantum_state(qstate)
        results.append( np.real(obsX.get_expectation_value(qstate)) )
        
    var = variance(results)
    return var

# Statistics

In [11]:
nqubits_list = [6,8,10,12,14]
depth_list = np.arange(6,318,6)
num_samp = 10000

for n in nqubits_list:
    var_list = []
    for d in depth_list:
        var_list.append( cal_var_sc(n, d, num_samp) )

    with open(f'data2/sc_n={n}_samp={num_samp}.csv', 'w') as f:
        writer = csv.writer(f)
        writer.writerow(depth_list)
        writer.writerow(var_list)